# LamoLLM - Google Colab

A 1B parameter decoder-only transformer language model built from scratch with PyTorch.

**Steps:**
1. Clone the repo
2. Install dependencies
3. Train the model (pick a config size)
4. Generate text

**Runtime -> Change runtime type:**
- **GPU**: T4 (free) or A100
- **TPU**: TPU v5e-1 (free)

## 1. Setup & Clone Repo

In [ ]:
!git clone https://github.com/arthurlamonattopro/LamoLLM.git
%cd /content/LamoLLM

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")

DEVICE = "cpu"

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    try:
        import torch_xla.core.xla_model as xm
        DEVICE = "tpu"
        print(f"TPU: {xm.get_xla_supported_devices(devkind='TPU')[0]}")
        print(f"TPU cores: {xm.xrt_world_size()}")
    except ImportError:
        print("No GPU or TPU detected. Using CPU.")

print(f"\nUsing device: {DEVICE}")

## 2. Training

Pick a config:
- **tiny** (~25M params) - quick test, runs in minutes
- **small** (~125M params) - moderate, ~30 min on T4
- **default** (~1.1B params) - full model, needs A100 or long run

Device is auto-detected (GPU > TPU > CPU).

**Quiet progress output:** one log line every **5%** of training (loss, ppl, lr, tok/s, ETA), and a checkpoint saved every **5%** (`checkpoints/lamollm_latest.pt`, rolling). The final model goes to `checkpoints/lamollm_final.pt`. Use `--keep_all_checkpoints` to retain every milestone file.

In [ ]:
# One log line per 5% + a checkpoint every 5% of the run
!python scripts/train.py --config tiny --epochs 1

# Options:
#   --checkpoint_every_pct 5     save cadence (% of run)
#   --log_every_pct 5            console log cadence (% of run)
#   --keep_all_checkpoints       keep lamollm_<pct>_step_<N>.pt files too
#   --no_eval                    skip validation-loss at each milestone

## 3. Generate Text

In [ ]:
!python scripts/generate.py --checkpoint checkpoints/lamollm_final.pt --prompt "Hello, I am" --max_tokens 100

## 4. Python API (Optional)

Use the generator directly in a cell.

In [ ]:
import sys
sys.path.insert(0, '/content/LamoLLM')

from inference.generator import LamoGenerator

generator = LamoGenerator.from_checkpoint('checkpoints/lamollm_final.pt')

output = generator.generate('The future of AI is', max_new_tokens=100)
print(output)

## 5. Interactive Chat

In [ ]:
import sys
sys.path.insert(0, '/content/LamoLLM')

from inference.generator import LamoGenerator

generator = LamoGenerator.from_checkpoint('checkpoints/lamollm_final.pt')

# Try a few prompts
prompts = [
    "Once upon a time",
    "The meaning of life is",
    "In a distant galaxy",
]

for prompt in prompts:
    print(f"Prompt: {prompt}")
    print(f"Output: {generator.generate(prompt, max_new_tokens=80)}")
    print("-" * 50)